# Tuần 6 — Social Engineering đa nhãn và Risk Fusion

Sáu bộ phân loại nhị phân TF-IDF + Logistic Regression (Binary Relevance). Nhãn 700 tin được gán bằng rules v2, chưa có chuyên gia rà soát. Điểm trên Validation là mức khớp nhãn tự động; không được trình bày như độ chính xác phát hiện Social Engineering thực tế.

Sửa các luật quá rộng tuần 3: không gán giả mạo chỉ vì có “quý khách”; không gán yêu cầu OTP chỉ vì tin thông báo chứa OTP; yêu cầu thông tin có cụm hành động và loại trừ phủ định gần. Đây vẫn là weak labels, chưa giải quyết đầy đủ ngữ nghĩa.

In [1]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "phishguard").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "phishguard").is_dir():
    raise FileNotFoundError("Mở notebook trong thư mục notebooks của gói dự án.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from phishguard.signals import LABELS, rule_evidence
from phishguard.engine import Engine
from phishguard.fusion import tune_reviewed_validation


In [2]:
report = json.loads((PROJECT_DIR / "reports/validation_results.json").read_text(encoding="utf-8"))
se = pd.read_csv(PROJECT_DIR / "data/social_engineering_700_v2.csv")
print("Số mẫu:", len(se))
display(se.groupby("split")[LABELS].sum())
display(pd.DataFrame(report["social_engineering_validation_weak_labels"]).T)


Số mẫu: 700


,impersonation,urgency_fear,reward_benefit,sensitive_info_request,financial_transaction,link_login_app_action
split,,,,,,
test,0,7,35,3,7,31
train,3,38,169,11,17,103
validation,2,3,40,3,3,32


,precision,recall,f1,fpr,pr_auc,brier,threshold,confusion_matrix,train_positives,validation_positives,label_status
impersonation,0.030303,0.5,0.057143,0.316832,0.040385,0.020198,0.05,"[[69, 32], [1, 1]]",3,2,weak_rule_agreement_not_gold_accuracy
urgency_fear,1.0,1.0,1.0,0.0,1.0,0.051752,0.52,"[[100, 0], [0, 3]]",38,3,weak_rule_agreement_not_gold_accuracy
reward_benefit,0.769231,1.0,0.869565,0.190476,0.939123,0.116559,0.42,"[[51, 12], [0, 40]]",169,40,weak_rule_agreement_not_gold_accuracy
sensitive_info_request,0.666667,0.666667,0.666667,0.01,0.605556,0.026367,0.45,"[[99, 1], [1, 2]]",11,3,weak_rule_agreement_not_gold_accuracy
financial_transaction,0.6,1.0,0.75,0.02,0.755556,0.025961,0.28,"[[98, 2], [0, 3]]",17,3,weak_rule_agreement_not_gold_accuracy
link_login_app_action,0.896552,0.8125,0.852459,0.042254,0.916967,0.093755,0.43,"[[68, 3], [6, 26]]",103,32,weak_rule_agreement_not_gold_accuracy


## Kết hợp điểm
Tin không có URL: dùng điểm văn bản. Chỉ có URL: dùng mô hình URL và ngưỡng URL. Nhiều URL: dùng max(P_url). Có cả hai: α×P_text + (1−α)×max(P_url).

Chưa có checkpoint XLM-R tuần 4 trong đầu vào: bản chạy dùng TF-IDF + LR nhị phân từ nhãn SMS nguồn. Nhãn nguồn 1 gộp spam/scam, nên phép kết hợp hiện tại là **thử nghiệm proxy**, không phải xác suất phishing đúng nghĩa. Tối ưu alpha và ngưỡng trên Validation SMS có URL thật; không ghép SMS với URL ngẫu nhiên. Dữ liệu dùng chọn alpha chưa có gold labels.

In [3]:
fusion = report["fusion_validation_proxy"]
print(json.dumps(fusion, ensure_ascii=False, indent=2))
grid = pd.read_csv(PROJECT_DIR / "reports/fusion_validation_grid.csv")
display(grid.sort_values(["f1","recall"], ascending=False).head(8))
if fusion["alpha"] == 1.0:
    print("Kết quả chọn alpha=1.0: nhánh URL chưa cải thiện F1 trong thử nghiệm proxy này. Không ép alpha khác để tạo kết quả đẹp.")

{
  "precision": 0.972972972972973,
  "recall": 0.9230769230769231,
  "f1": 0.9473684210526315,
  "fpr": 0.011583011583011582,
  "pr_auc": 0.9839585391678176,
  "brier": 0.04840013284953046,
  "threshold": 0.5199999999999999,
  "confusion_matrix": [
    [
      256,
      3
    ],
    [
      9,
      108
    ]
  ],
  "alpha": 1.0,
  "status": "proxy_only_not_validated_phishing_probability",
  "target": "source_spam_or_scam",
  "paired_validation_rows": 236,
  "validation_rows": 376,
  "test_used": false
}
Kết quả chọn alpha=1.0: nhánh URL chưa cải thiện F1 trong thử nghiệm proxy này. Không ép alpha khác để tạo kết quả đẹp.


,precision,recall,f1,fpr,pr_auc,brier,threshold,confusion_matrix,alpha,status,target,paired_validation_rows,validation_rows,test_used
20,0.972973,0.923077,0.947368,0.011583,0.983959,0.048400,0.52,"[[256, 3], [9, 108]]",1.00,proxy_only_not_validated_phishing_probability,source_spam_or_scam,236.0,376.0,False
19,0.964286,0.923077,0.943231,0.015444,0.982720,0.051012,0.54,"[[255, 4], [9, 108]]",0.95,NaN,NaN,NaN,NaN,NaN
17,0.940171,0.940171,0.940171,0.027027,0.979756,0.061794,0.55,"[[252, 7], [7, 110]]",0.85,NaN,NaN,NaN,NaN,NaN
18,0.955752,0.923077,0.939130,0.019305,0.982039,0.055476,0.54,"[[254, 5], [9, 108]]",0.90,NaN,NaN,NaN,NaN,NaN
16,0.916667,0.940171,0.928270,0.038610,0.975184,0.069965,0.53,"[[249, 10], [7, 110]]",0.80,NaN,NaN,NaN,NaN,NaN
15,0.909091,0.940171,0.924370,0.042471,0.968534,0.079988,0.52,"[[248, 11], [7, 110]]",0.75,NaN,NaN,NaN,NaN,NaN
14,0.888000,0.948718,0.917355,0.054054,0.954569,0.091866,0.50,"[[245, 14], [6, 111]]",0.70,NaN,NaN,NaN,NaN,NaN
13,0.874016,0.948718,0.909836,0.061776,0.945143,0.105596,0.48,"[[243, 16], [6, 111]]",0.65,NaN,NaN,NaN,NaN,NaN


In [4]:
engine = Engine(PROJECT_DIR / "models")
result = engine.analyze("Vui lòng đăng nhập để nhận quà tại https://example.com/login")
print({key: result[key] for key in ["mode","risk_score","threshold","alpha","p_text_spam_scam","p_url_phishing"]})
display(pd.DataFrame(result["signals"])[["key","probability","predicted","evidence"]])

{'mode': 'fusion_proxy', 'risk_score': 0.8553213899112513, 'threshold': 0.5199999999999999, 'alpha': 1.0, 'p_text_spam_scam': 0.8553213899112513, 'p_url_phishing': 0.9998670816421509}


,key,probability,predicted,evidence
0,impersonation,0.033560,None,[]
1,urgency_fear,0.160384,None,[]
2,reward_benefit,0.439803,True,"[{'start': 22, 'end': 30, 'text': 'nhận quà'}]"
3,sensitive_info_request,0.149044,None,[]
4,financial_transaction,0.106714,None,[]
5,link_login_app_action,0.488140,True,"[{'start': 9, 'end': 18, 'text': 'đăng nhập'}]"


## Tối ưu lại khi có nhãn người rà soát
Điền CSV theo mẫu `docs/paired_validation_template.csv`, dùng xác suất từ mô hình văn bản tuần 4 và URL model không học các nhóm validation đó. `split` phải là validation, `label_status` phải là human_reviewed, `y_phishing` 0/1 riêng với spam. Hàm chặn Test và weak labels; người nghiên cứu vẫn phải xác minh nguồn dữ liệu/prediction. Kết quả không tự động thay mô hình phục vụ khi mục tiêu nhãn đã thay đổi.

In [5]:
REVIEWED_VALIDATION_CSV = None  # ví dụ PROJECT_DIR / "data/paired_validation_reviewed.csv"
if REVIEWED_VALIDATION_CSV is not None:
    gold_result = tune_reviewed_validation(REVIEWED_VALIDATION_CSV, PROJECT_DIR / "reports/fusion_gold_validation.json")
    display(gold_result["selected"])
else:
    print("Chưa có nhãn phishing đã rà soát: giữ nguyên trạng thái proxy của thử nghiệm hiện tại.")

Chưa có nhãn phishing đã rà soát: giữ nguyên trạng thái proxy của thử nghiệm hiện tại.
